# YOLO Inference Notebook (Colab)
This notebook installs dependencies, prepares a small example dataset (optional), runs inference with `ultralytics` (YOLOv8), and assembles a short MP4 of the results.

Instructions:
- Upload the `week-05/` folder to Colab, mount Google Drive, or modify the `DATA_ROOT` path below to point to your files.
- Run cells in order.

In [ ]:
# Install dependencies
!pip install -U ultralytics pillow opencv-python-headless ffmpeg-python --quiet

In [ ]:
# Ensure ffmpeg exists (Colab usually has it)
import shutil, os
if shutil.which('ffmpeg') is None:
    print('ffmpeg not found, installing...')
    !apt-get update -y && apt-get install -y ffmpeg
else:
    print('ffmpeg found at', shutil.which('ffmpeg'))

## Data location
Set `DATA_ROOT` to the folder that contains `images_resized/` and `labels/` from the repository. If you uploaded the repo to Colab's working directory, use the local path. You can also mount Google Drive and set `DATA_ROOT` accordingly.

In [ ]:
# Optionally mount Google Drive (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')

# By default assume the notebook is run from the repo root in Colab (upload week-05 folder)
DATA_ROOT = '/content/week-05'  # change if needed
import os
os.makedirs(DATA_ROOT, exist_ok=True)
print('Data root set to', DATA_ROOT)

## (Optional) Create a tiny example dataset inside Colab
Run this cell if you don't want to upload images and labels — it will create two images and YOLO-format labels so you can test the pipeline end-to-end.

In [ ]:
# Create example dataset (2 images, train/val)
from PIL import Image, ImageDraw
import os
images_train = os.path.join(DATA_ROOT, 'images', 'train')
images_val = os.path.join(DATA_ROOT, 'images', 'val')
labels_train = os.path.join(DATA_ROOT, 'labels', 'train')
labels_val = os.path.join(DATA_ROOT, 'labels', 'val')
for p in [images_train, images_val, labels_train, labels_val, os.path.join(DATA_ROOT, 'images_resized')]:
    os.makedirs(p, exist_ok=True)

# Create train image
img = Image.new('RGB', (640, 480), color=(73, 109, 137))
draw = ImageDraw.Draw(img)
draw.rectangle([120,80,400,320], outline='red', width=4)
img.save(os.path.join(images_train, 'img_train.jpg'))
# YOLO label: class x_center y_center w h (normalized)
w, h = img.size
x_center = (120+400)/2/w
y_center = (80+320)/2/h
bw = (400-120)/w
bh = (320-80)/h
with open(os.path.join(labels_train, 'img_train.txt'), 'w') as f:
    f.write(f'0 {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f}
')

# Create val image (smaller)
img2 = Image.new('RGB', (512, 384), color=(120, 80, 50))
draw2 = ImageDraw.Draw(img2)
draw2.rectangle([60,40,260,200], outline='green', width=4)
img2.save(os.path.join(images_val, 'img_val.jpg'))
w2,h2 = img2.size
x_center2 = (60+260)/2/w2
y_center2 = (40+200)/2/h2
bw2 = (260-60)/w2
bh2 = (200-40)/h2
with open(os.path.join(labels_val, 'img_val.txt'), 'w') as f2:
    f2.write(f'0 {x_center2:.6f} {y_center2:.6f} {bw2:.6f} {bh2:.6f}
')

# Copy images to images_resized (ultralytics will accept them as source)
import shutil
shutil.copy(os.path.join(images_train, 'img_train.jpg'), os.path.join(DATA_ROOT, 'images_resized', 'img_train.jpg'))
shutil.copy(os.path.join(images_val, 'img_val.jpg'), os.path.join(DATA_ROOT, 'images_resized', 'img_val.jpg'))
print('Example dataset created under', DATA_ROOT)

## Run YOLO inference (ultralytics)
This will download `yolov8n.pt` automatically if not present and save outputs to `outputs/detections_ultralytics/`.

In [ ]:
from ultralytics import YOLO
import os
src = os.path.join(DATA_ROOT, 'images_resized')
print('Running inference on', src)
model = YOLO('yolov8n.pt')
# predict with saving annotated images to outputs/detections_ultralytics
res = model.predict(source=src, imgsz=384, conf=0.25, save=True, project=os.path.join(DATA_ROOT, 'outputs'), name='detections_ultralytics')
print('Inference finished. Check', os.path.join(DATA_ROOT, 'outputs','detections_ultralytics'))

## Create a short MP4 from the annotated images
This cell uses `ffmpeg` to create `outputs/detections_video.mp4` and a silent audio track then muxes into `outputs/detections_video_audio.mp4`.

In [ ]:
import glob, subprocess, os
annot_dir = os.path.join(DATA_ROOT, 'outputs', 'detections_ultralytics')
imgs = sorted(glob.glob(os.path.join(annot_dir, '*.jpg')))
if len(imgs) == 0:
    print('No annotated images found in', annot_dir)
else:
    out_video = os.path.join(DATA_ROOT, 'outputs', 'detections_video.mp4')
    final_out = os.path.join(DATA_ROOT, 'outputs', 'detections_video_audio.mp4')
    os.makedirs(os.path.dirname(out_video), exist_ok=True)
    # Create slideshow video (1s per image)
    cmd = [ 'ffmpeg', '-y', '-loop', '1', '-t', '1', '-i', imgs[0], '-loop', '1', '-t', '1', '-i', imgs[1] if len(imgs)>1 else imgs[0], '-filter_complex', '[0:v][1:v]concat=n=2:v=1:a=0,format=yuv420p', '-c:v', 'libx264', out_video ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    # Create silent audio and mux
    silent = os.path.join(DATA_ROOT, 'silent.wav')
    subprocess.run(['ffmpeg','-y','-f','lavfi','-i','anullsrc=channel_layout=stereo:sample_rate=44100','-t','2',silent], check=True)
    subprocess.run(['ffmpeg','-y','-i', out_video, '-i', silent, '-c:v','copy','-c:a','aac','-shortest', final_out], check=True)
    print('Created', final_out)

## Download outputs
Use the Colab file browser or the following commands to download outputs to your machine or copy them to Google Drive. Example: `files.download('/content/week-05/outputs/detections_video_audio.mp4')`